In [ ]:
# Cell 1: Imports
import os
import ast
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from shapely import wkt
from tqdm import tqdm
import matplotlib.pyplot as plt

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Cell 2: Config
class Config:
    CSV_PATH = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/SummaryData/SN6_Train_AOI_11_Rotterdam_Buildings.csv' # Replace with your CSV path
    OPTICAL_DIR = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/PS-RGB' # Replace with Optical dir
    SAR_DIR = '/kaggle/input/datasets/sandhiwangiyana/spacenet-6-multisensor-allweather-mapping/AOI_11_Rotterdam/SAR-Intensity'         # Replace with SAR dir
    
    IMG_SIZE = 512 # As per SpaceNet 6 standards mentioned in the paper
    BATCH_SIZE = 4
    EPOCHS = 10
    LR = 1e-4
    NUM_SAMPLES = 100 # Limit for quick training/testing
    
    # Loss weights as defined in the paper
    LAMBDA_FOOTPRINT = 1.0
    LAMBDA_HEIGHT = 0.5

In [ ]:
# Cell 3: Dataset Class
class SpaceNetDataset(Dataset):
    def __init__(self, csv_path, optical_dir, sar_dir, img_size=512, limit=None):
        self.optical_dir = optical_dir
        self.sar_dir = sar_dir
        self.img_size = img_size
        
        # Load and group data by ImageId
        df = pd.read_csv(csv_path)
        self.grouped = df.groupby('ImageId')
        self.image_ids = list(self.grouped.groups.keys())
        
        if limit:
            self.image_ids = self.image_ids[:limit]
            
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size, img_size), antialias=True)
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        group = self.grouped.get_group(img_id)
        
        # 1. Load Images (Assuming .tif or .png formats, adjust if necessary)
        opt_path = os.path.join(self.optical_dir, f"{img_id}.png") # Adjust extension
        sar_path = os.path.join(self.sar_dir, f"{img_id}.png")     # Adjust extension
        
        # Dummy fallback if file doesn't exist for testing purposes
        if not os.path.exists(opt_path):
            opt_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            sar_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8) # Or 1 channel if SAR is grayscale
        else:
            opt_img = cv2.imread(opt_path)[..., ::-1] # BGR to RGB
            sar_img = cv2.imread(sar_path, cv2.IMREAD_GRAYSCALE)
            if len(sar_img.shape) == 2:
                sar_img = np.expand_dims(sar_img, axis=-1)
                
        # 2. Generate Ground Truth Masks
        mask_footprint = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        mask_height = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        
        for _, row in group.iterrows():
            poly_wkt = row['PolygonWKT_Pix']
            height = row['Mean_Building_Height']
            
            if pd.isna(poly_wkt) or poly_wkt.strip() == 'POLYGON EMPTY':
                continue
                
            try:
                poly = wkt.loads(poly_wkt)
                # Extract coordinates and scale if necessary (assuming coords are pixel relative to 512x512)
                coords = np.array(poly.exterior.coords, dtype=np.int32)
                
                # Fill footprint mask with 1s
                cv2.fillPoly(mask_footprint, [coords], 1.0)
                # Fill height mask with the continuous height values
                cv2.fillPoly(mask_height, [coords], float(height))
            except Exception as e:
                pass # Handle malformed WKTs
                
        # 3. Convert to Tensors
        opt_tensor = self.transform(opt_img)
        # Ensure SAR has 3 channels for pre-trained weights, or adjust first layer
        if sar_img.shape[-1] == 1:
            sar_img = np.repeat(sar_img, 3, axis=-1)
        sar_tensor = self.transform(sar_img)
        
        mask_footprint = torch.tensor(mask_footprint, dtype=torch.long) # CrossEntropy expects long
        mask_height = torch.tensor(mask_height, dtype=torch.float32).unsqueeze(0)
        
        return opt_tensor, sar_tensor, mask_footprint, mask_height

In [ ]:
# Cell 4: Modules (Encoders & Fusion)
class PretrainedCNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        self.initial = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool) # F0
        self.layer1 = resnet.layer1 # F1
        self.layer2 = resnet.layer2 # F2
        self.layer3 = resnet.layer3 # F3
        self.layer4 = resnet.layer4 # Fout (1/32 resolution, we will upsample to 1/16 to match paper)
        
        # The paper uses 1/16 as the lowest resolution. We'll adjust layer4 output.
        self.up_align = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)

    def forward(self, x):
        f0 = self.initial(x)  # H/4 (ResNet skips H/2 due to initial conv stride + maxpool)
        f1 = self.layer1(f0)  # H/4
        f2 = self.layer2(f1)  # H/8
        f3 = self.layer3(f2)  # H/16
        f4 = self.layer4(f3)  # H/32
        f_out = self.up_align(f4) # H/16, Channels: 256
        return f0, f1, f2, f_out

class CrossFusionBlock(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()
        self.norm1_sar = nn.LayerNorm(embed_dim)
        self.norm1_opt = nn.LayerNorm(embed_dim)
        
        self.cross_attn_sar = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.cross_attn_opt = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        self.norm2_sar = nn.LayerNorm(embed_dim)
        self.norm2_opt = nn.LayerNorm(embed_dim)
        
        self.mlp_sar = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Linear(embed_dim * 2, embed_dim)
        )
        self.mlp_opt = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(), nn.Linear(embed_dim * 2, embed_dim)
        )

    def forward(self, sar_feat, opt_feat):
        # Reshape (B, C, H, W) -> (B, H*W, C)
        B, C, H, W = sar_feat.shape
        sar_flat = sar_feat.flatten(2).transpose(1, 2)
        opt_flat = opt_feat.flatten(2).transpose(1, 2)
        
        # Norm
        sar_norm = self.norm1_sar(sar_flat)
        opt_norm = self.norm1_opt(opt_flat)
        
        # Cross Attention: Modality acts as Query, the OTHER modality acts as Key/Value
        attn_sar, _ = self.cross_attn_sar(sar_norm, opt_norm, opt_norm)
        sar_out = sar_flat + attn_sar
        
        attn_opt, _ = self.cross_attn_opt(opt_norm, sar_norm, sar_norm)
        opt_out = opt_flat + attn_opt
        
        # MLP
        sar_out = sar_out + self.mlp_sar(self.norm2_sar(sar_out))
        opt_out = opt_out + self.mlp_opt(self.norm2_opt(opt_out))
        
        # Reshape back to (B, C, H, W)
        sar_out = sar_out.transpose(1, 2).view(B, C, H, W)
        opt_out = opt_out.transpose(1, 2).view(B, C, H, W)
        
        return sar_out, opt_out

In [ ]:
# Cell 5: Decoder and Full Architecture
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Ensure spatial dimensions match if rounding issues occur
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class FusionHeightNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_opt = PretrainedCNNEncoder()
        self.enc_sar = PretrainedCNNEncoder()
        
        self.cross_fusion = CrossFusionBlock(embed_dim=256)
        
        # Joint fusion of optical and SAR for the shared lowest resolution
        self.fusion_conv = nn.Conv2d(512, 256, 1) 
        
        # Footprint Decoder (using optical skips for sharp edges)
        self.foot_dec1 = DecoderBlock(256, 128, 128) # Uses f2
        self.foot_dec2 = DecoderBlock(128, 64, 64)   # Uses f1
        self.foot_dec3 = DecoderBlock(64, 64, 64)    # Uses f0
        self.foot_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Upsample(scale_factor=4, mode='bilinear'), # Upscale remaining H/4 to H
            nn.Conv2d(32, 2, 1) # 2 Classes: Background, Footprint
        )
        
        # Height Decoder (using SAR skips for structural geometry)
        self.height_dec1 = DecoderBlock(256, 128, 128)
        self.height_dec2 = DecoderBlock(128, 64, 64)
        self.height_dec3 = DecoderBlock(64, 64, 64)
        self.height_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Upsample(scale_factor=4, mode='bilinear'),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid() # Restricts output to 0-1 (Assumes GT height is normalized!)
        )

    def forward(self, opt, sar):
        o_f0, o_f1, o_f2, o_fout = self.enc_opt(opt)
        s_f0, s_f1, s_f2, s_fout = self.enc_sar(sar)
        
        # Multi-Level Cross Fusion at the bottleneck
        s_fused, o_fused = self.cross_fusion(s_fout, o_fout)
        
        # Combine for decoder input
        f_fusion = self.fusion_conv(torch.cat([o_fused, s_fused], dim=1))
        
        # Footprint Branch
        d_foot = self.foot_dec1(f_fusion, o_f2)
        d_foot = self.foot_dec2(d_foot, o_f1)
        d_foot = self.foot_dec3(d_foot, o_f0)
        pred_footprint = self.foot_head(d_foot)
        
        # Height Branch
        d_height = self.height_dec1(f_fusion, s_f2)
        d_height = self.height_dec2(d_height, s_f1)
        d_height = self.height_dec3(d_height, s_f0)
        base_height = self.height_head(d_height)
        
        # Semantic Refinement (Paper Eq 16)
        # Instead of non-differentiable argmax, we use the probability of the footprint class
        footprint_prob = F.softmax(pred_footprint, dim=1)[:, 1:2, :, :] 
        pred_height = base_height * footprint_prob
        
        return pred_footprint, pred_height

In [ ]:
# Cell 6: Losses
def dice_loss(pred, target, smooth=1e-6):
    pred = F.softmax(pred, dim=1)[:, 1, :, :] # Take footprint class prob
    intersection = (pred * target).sum(dim=(1, 2))
    union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

def fusion_loss(pred_foot, pred_height, gt_foot, gt_height, config):
    # Footprint Loss (CrossEntropy + Dice)
    ce_loss = F.cross_entropy(pred_foot, gt_foot)
    d_loss = dice_loss(pred_foot, gt_foot)
    L_footprint = ce_loss + d_loss
    
    # Height Loss (Smooth L1)
    # Note: Ensure your gt_height is normalized between 0-1 for this to work correctly with Sigmoid
    # E.g., gt_height = gt_height / MAX_HEIGHT
    L_height = F.smooth_l1_loss(pred_height, gt_height)
    
    # Total
    L_total = (config.LAMBDA_FOOTPRINT * L_footprint) + (config.LAMBDA_HEIGHT * L_height)
    return L_total, L_footprint, L_height

In [ ]:
# Cell 7: Execution
if __name__ == '__main__':
    # Initialize Dataset
    dataset = SpaceNetDataset(
        csv_path=Config.CSV_PATH, 
        optical_dir=Config.OPTICAL_DIR, 
        sar_dir=Config.SAR_DIR, 
        img_size=Config.IMG_SIZE,
        limit=Config.NUM_SAMPLES
    )
    
    dataloader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    
    # Initialize Model
    model = FusionHeightNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=Config.LR, momentum=0.9, weight_decay=1e-4)
    
    # Optional: Max height for normalization (Update this based on your dataset stats)
    MAX_HEIGHT = 50.0 
    
    print("Starting Training...")
    for epoch in range(Config.EPOCHS):
        model.train()
        epoch_loss = 0.0
        
        progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        for opt, sar, gt_foot, gt_height in progress:
            opt = opt.to(device)
            sar = sar.to(device)
            gt_foot = gt_foot.to(device)
            
            # Normalize GT Height for Sigmoid compatibility
            gt_height = (gt_height / MAX_HEIGHT).to(device)
            
            optimizer.zero_grad()
            
            pred_foot, pred_height = model(opt, sar)
            
            loss, l_foot, l_height = fusion_loss(pred_foot, pred_height, gt_foot, gt_height, Config)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            progress.set_postfix({'Total Loss': loss.item(), 'Foot Loss': l_foot.item(), 'Height Loss': l_height.item()})
            
        avg_loss = epoch_loss/len(dataloader)
        print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")
        
        # --- NEW: Save after every epoch ---
        save_path = f'fusion_height_net_epoch_{epoch+1}.pth'
        torch.save(model.state_dict(), save_path)
        print(f"Saved model checkpoint to {save_path}")
        # -----------------------------------
        
    print("Training Complete.")